# SymbolicEngineWithLLM — Standalone Runner (Exp-2 Method 5)
**HypatiaX Research · RF-09**

Runs `SymbolicEngineWithLLM` (PySR + LLM tool loop) across all 30 Feynman equations
from `ExperimentProtocolAll`.  Each equation runs in its **own isolated subprocess**
so Julia's JIT heap is fully released between equations.

| Setting | Value |
|---|---|
| PySR timeout | 1100 s/equation |
| Proc timeout | timeout + 120 s |
| Parallelism | **multithreading** (fixed from multiprocessing — Distributed.ProcessExitedException on Kaggle/Julia 1.11) |
| Resume | `--resume` flag (or re-run cell — checkpoint survives) |
| Runtime estimate | ~12–24 h on 4-vCPU Kaggle CPU notebook |

> **Recommended runtime:** Settings → Accelerator → **None** (4 vCPUs).
> GPU notebooks give only 2 CPUs — worse for PySR.

> **Setup order:** Run Cell 1 FIRST, then Cell 2 (installs Julia + PySR).

> **API key:** Kaggle Secrets → `ANTHROPIC_API_KEY`.


In [ ]:
# CELL 1 — Run this BEFORE installing PySR
import os, multiprocessing
n_cores = multiprocessing.cpu_count()
print(f'Kaggle CPU cores available: {n_cores}')
os.environ['JULIA_NUM_THREADS']               = str(n_cores)
os.environ['JULIA_EXCLUSIVE']                 = '0'
os.environ['PYTHON_JULIACALL_HANDLE_SIGNALS'] = 'yes'
os.environ['LLM_MODEL'] = 'claude-sonnet-4-20250514'
print(f"JULIA_NUM_THREADS set to {os.environ['JULIA_NUM_THREADS']} OK")
print(f"LLM_MODEL = {os.environ['LLM_MODEL']}")


In [ ]:
import subprocess, sys, os
JULIA_VERSION = '1.11.4'
JULIA_MINOR   = '1.11'
print(f'Downloading Julia {JULIA_VERSION}...')
tarball = f'julia-{JULIA_VERSION}-linux-x86_64.tar.gz'
url = (f'https://julialang-s3.julialang.org/bin/linux/x64/{JULIA_MINOR}/{tarball}')
subprocess.run(['wget', '-q', url], check=True)
subprocess.run(['tar', '-xzf', tarball], check=True)
julia_bin = f'/hypatiax/data/results/julia-{JULIA_VERSION}/bin'
os.environ['PATH'] = julia_bin + ':' + os.environ.get('PATH', '')
print(f'Julia {JULIA_VERSION} installed OK')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'pysr', 'scikit-learn', 'scipy', 'numpy', 'pandas', 'anthropic'], check=True)
import pysr
print(f'PySR version: {pysr.__version__}')
result = subprocess.run(['julia', '-e', 'println(Threads.nthreads())'],
                        capture_output=True, text=True, timeout=120)
print(f'Julia threads: {result.stdout.strip()}')


In [ ]:
import os, time, json, warnings, gc, sys, subprocess, tempfile, textwrap, signal
import argparse
from datetime import datetime
from pathlib import Path
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from scipy import stats as scipy_stats
warnings.filterwarnings('ignore')
try:
    from pysr import PySRRegressor
    PYSR_AVAILABLE = True
    print('PySR loaded OK')
except ImportError:
    PYSR_AVAILABLE = False
    print('ERROR: pysr not installed — run Cell 2 first')
_PYSR_VALID_PARAMS = None


In [ ]:
# Kaggle Secrets: Settings → Add-ons → Secrets → Add ANTHROPIC_API_KEY
import os
try:
    from kaggle_secrets import UserSecretsClient
    ANTHROPIC_API_KEY = UserSecretsClient().get_secret('ANTHROPIC_API_KEY')
    print('API key loaded from Kaggle Secrets OK')
except Exception as e:
    print(f'Kaggle Secrets unavailable: {e}')
    ANTHROPIC_API_KEY = ''  # paste key here only as last resort
os.environ['ANTHROPIC_API_KEY'] = ANTHROPIC_API_KEY or ''
if ANTHROPIC_API_KEY and ANTHROPIC_API_KEY.startswith('sk-ant-'):
    print('API key set OK')
else:
    print('Key missing — add via Kaggle Secrets')


In [ ]:
import os
OUTPUT_DIR       = '/hypatiax/data/results'
LOG_DIR          = '/hypatiax/logs'
os.makedirs(LOG_DIR, exist_ok=True)

CHECKPOINT_PATH  = os.path.join(LOG_DIR, 'exp2_symbolic_engine_checkpoint.json')
PYSR_TIMEOUT     = int(os.environ.get('PYSR_TIMEOUT', 1100))
METHOD_TIMEOUT   = 1200   # proc_timeout = PYSR_TIMEOUT + 120
PASS_THRESHOLD   = 9
NUM_SAMPLES      = 200

CFG = dict(
    pysr_timeout   = PYSR_TIMEOUT,
    num_samples    = NUM_SAMPLES,
    pass_threshold = PASS_THRESHOLD,
    checkpoint     = CHECKPOINT_PATH,
    resume         = True,   # set False to start fresh
    one_equation   = None,   # e.g. 'Kinetic' for a single-equation test
    ram_limit_gb   = 0,      # e.g. 12 on a 16 GB machine; 0 = no limit
)
print('Config:', CFG)
print(f"Checkpoint → {CHECKPOINT_PATH}")


In [ ]:
# ── ExperimentProtocolAll (v4.0) — inline, no external import needed ─────────
# Paste the full class here so the notebook is truly self-contained.
# Source: experiment_protocol_all_30.py

import numpy as np

class ExperimentProtocolAll:
    """Complete protocol with all 30 test cases — v4.0 BEST OF BOTH"""

    @staticmethod
    def get_all_domains():
        return [
            'mechanics', 'thermodynamics', 'electromagnetism',
            'fluid_dynamics', 'optics', 'quantum',
            'chemistry', 'biology', 'mathematics', 'economics',
        ]

    @staticmethod
    def load_test_data(domain, num_samples=300):
        np.random.seed(42)
        test_cases = []
        N = num_samples

        if domain == 'mechanics':
            m = np.random.uniform(0.1, 10, N); v = np.random.uniform(0.1, 50, N)
            test_cases.append(('Kinetic Energy: KE = (1/2)*m*v²', np.column_stack([m,v]), 0.5*m*v**2, ['m','v'],
                {'equation_name':'kinetic_energy','difficulty':'easy','formula_type':'power_law','ground_truth':'0.5 * m * v**2','variable_descriptions':{'m':'Object mass','v':'Object velocity'},'protocol':'A'}))
            m = np.random.uniform(0.1,100,N); g = np.random.uniform(9.7,9.9,N); h = np.random.uniform(0,100,N)
            test_cases.append(('Gravitational Potential Energy: PE = m*g*h', np.column_stack([m,g,h]), m*g*h, ['m','g','h'],
                {'equation_name':'gravitational_potential_energy','difficulty':'easy','formula_type':'product','ground_truth':'m * g * h','variable_descriptions':{'m':'Mass','g':'Gravity','h':'Height'},'protocol':'A'}))
            k = np.random.uniform(1,100,N); x = np.random.uniform(-2,2,N)
            test_cases.append(("Hooke's Law: F = k*x", np.column_stack([k,x]), k*x, ['k','x'],
                {'equation_name':'hookes_law','difficulty':'easy','formula_type':'linear','ground_truth':'k * x','variable_descriptions':{'k':'Spring constant','x':'Displacement'},'protocol':'A'}))

        elif domain == 'thermodynamics':
            n = np.random.uniform(0.1,10,N); T = np.random.uniform(200,400,N); V = np.random.uniform(0.01,1,N)
            test_cases.append(('Ideal Gas Law: PV = nRT => P = n*8.314*T/V', np.column_stack([n,T,V]), n*8.314*T/V, ['n','T','V'],
                {'equation_name':'ideal_gas_law','difficulty':'medium','formula_type':'algebraic','ground_truth':'n * 8.314 * T / V','variable_descriptions':{'n':'moles','T':'temperature','V':'volume'},'protocol':'A'}))
            m = np.random.uniform(0.1,10,N); c = np.random.uniform(100,5000,N); dT = np.random.uniform(1,100,N)
            test_cases.append(('Heat Capacity: Q = m*c*ΔT', np.column_stack([m,c,dT]), m*c*dT, ['m','c','dT'],
                {'equation_name':'heat_capacity','difficulty':'easy','formula_type':'product','ground_truth':'m * c * dT','variable_descriptions':{'m':'mass','c':'specific heat','dT':'temp change'},'protocol':'A'}))
            Tc = np.random.uniform(200,300,N); Th = np.random.uniform(400,600,N)
            test_cases.append(('Carnot Efficiency: η = 1 - Tc/Th', np.column_stack([Tc,Th]), 1-Tc/Th, ['Tc','Th'],
                {'equation_name':'carnot_efficiency','difficulty':'easy','formula_type':'algebraic','ground_truth':'1 - Tc / Th','variable_descriptions':{'Tc':'cold temp','Th':'hot temp'},'protocol':'A'}))

        elif domain == 'electromagnetism':
            q1 = np.random.uniform(1e-9,1e-6,N); q2 = np.random.uniform(1e-9,1e-6,N); r = np.random.uniform(0.01,1,N)
            test_cases.append(("Coulomb's Law: F = 8.99e9*q1*q2/r²", np.column_stack([q1,q2,r]), 8.99e9*q1*q2/r**2, ['q1','q2','r'],
                {'equation_name':'coulomb_law','difficulty':'medium','formula_type':'power_law','ground_truth':'8.99e9 * q1 * q2 / r**2','variable_descriptions':{'q1':'charge 1','q2':'charge 2','r':'distance'},'protocol':'A'}))
            I = np.random.uniform(0.1,10,N); R = np.random.uniform(1,1000,N)
            test_cases.append(("Ohm's Law: V = I*R", np.column_stack([I,R]), I*R, ['I','R'],
                {'equation_name':'ohms_law','difficulty':'easy','formula_type':'linear','ground_truth':'I * R','variable_descriptions':{'I':'current','R':'resistance'},'protocol':'A'}))
            q = np.random.uniform(1e-9,1e-6,N); v = np.random.uniform(1,100,N); B = np.random.uniform(0.1,10,N)
            test_cases.append(('Lorentz Force: F = q*v*B', np.column_stack([q,v,B]), q*v*B, ['q','v','B'],
                {'equation_name':'lorentz_force','difficulty':'easy','formula_type':'product','ground_truth':'q * v * B','variable_descriptions':{'q':'charge','v':'velocity','B':'B-field'},'protocol':'A'}))

        elif domain == 'fluid_dynamics':
            P = np.random.uniform(1e5,2e5,N); rho = np.random.uniform(800,1200,N); v = np.random.uniform(0.1,15,N); g = np.random.uniform(9.6,9.9,N); h = np.random.uniform(0,10,N)
            test_cases.append(("Bernoulli's Equation: Total = P + (1/2)*ρ*v² + ρ*g*h", np.column_stack([P,rho,v,g,h]), P+0.5*rho*v**2+rho*g*h, ['P','rho','v','g','h'],
                {'equation_name':'bernoulli_equation','difficulty':'hard','formula_type':'additive_polynomial','ground_truth':'P + 0.5 * rho * v**2 + rho * g * h','variable_descriptions':{'P':'pressure','rho':'density','v':'velocity','g':'gravity','h':'height'},'protocol':'A'}))
            rho = np.random.uniform(800,1200,N); v = np.random.uniform(0.1,10,N); L = np.random.uniform(0.01,1,N); mu = np.random.uniform(0.001,0.1,N)
            test_cases.append(('Reynolds Number: Re = ρ*v*L/μ', np.column_stack([rho,v,L,mu]), rho*v*L/mu, ['rho','v','L','mu'],
                {'equation_name':'reynolds_number','difficulty':'easy','formula_type':'algebraic','ground_truth':'rho * v * L / mu','variable_descriptions':{'rho':'density','v':'velocity','L':'length','mu':'viscosity'},'protocol':'A'}))
            dP = np.random.uniform(100,10000,N); r = np.random.uniform(0.001,0.1,N); L = np.random.uniform(0.1,10,N)
            test_cases.append(('Hagen-Poiseuille: Q = π*r⁴*ΔP/(8*0.001*L)', np.column_stack([dP,r,L]), (np.pi*r**4*dP)/(8*0.001*L), ['dP','r','L'],
                {'equation_name':'hagen_poiseuille','difficulty':'hard','formula_type':'power_law','ground_truth':'(np.pi * r**4 * dP) / (8 * 0.001 * L)','variable_descriptions':{'dP':'pressure diff','r':'pipe radius','L':'pipe length'},'protocol':'A'}))

        elif domain == 'optics':
            do = np.random.uniform(0.1,10,N); di = np.random.uniform(0.1,10,N)
            test_cases.append(('Thin Lens: 1/f = 1/do + 1/di', np.column_stack([do,di]), 1/do+1/di, ['do','di'],
                {'equation_name':'thin_lens_equation','difficulty':'easy','formula_type':'algebraic','ground_truth':'1/do + 1/di','variable_descriptions':{'do':'object dist','di':'image dist'},'protocol':'A'}))
            n1 = np.random.uniform(1,2.5,N); st = np.random.uniform(0.1,0.9,N)
            test_cases.append(("Snell's Law: n1*sin(θ1) = n2*sin(θ2)", np.column_stack([n1,st]), n1*st, ['n1','sin_theta1'],
                {'equation_name':'snells_law','difficulty':'easy','formula_type':'linear','ground_truth':'n1 * sin_theta1','variable_descriptions':{'n1':'refractive index','sin_theta1':'sin of angle'},'protocol':'A'}))
            wl = np.random.uniform(400e-9,700e-9,N); a = np.random.uniform(1e-6,1e-4,N)
            test_cases.append(('Diffraction: sin(θ) = λ/a', np.column_stack([wl,a]), wl/a, ['wavelength','a'],
                {'equation_name':'single_slit_diffraction','difficulty':'easy','formula_type':'algebraic','ground_truth':'wavelength / a','variable_descriptions':{'wavelength':'wavelength','a':'slit width'},'protocol':'A'}))

        elif domain == 'quantum':
            f = np.random.uniform(4e14,7.5e14,N)
            test_cases.append(('Photon Energy: E = 4.136e-15*f (visible light, eV units)', f.reshape(-1,1), 4.136e-15*f, ['f'],
                {'equation_name':'photon_energy','difficulty':'easy','formula_type':'linear','ground_truth':'4.136e-15 * f','variable_descriptions':{'f':'frequency'},'protocol':'A'}))
            v = np.random.uniform(100,10000,N)
            test_cases.append(('de Broglie Wavelength: λ = 1/(v) (normalized h=m=1)', v.reshape(-1,1), 1.0/v, ['v'],
                {'equation_name':'de_broglie_wavelength','difficulty':'easy','formula_type':'algebraic','ground_truth':'1.0 / v','variable_descriptions':{'v':'velocity (km/s)'},'protocol':'A'}))
            ct = np.random.uniform(-1,1,N); cw = 6.626e-34/(9.109e-31*3e8)
            test_cases.append(('Compton Shift: Δλ = 2.426e-12*(1-cos(θ))', ct.reshape(-1,1), cw*(1-ct), ['cos_theta'],
                {'equation_name':'compton_shift','difficulty':'medium','formula_type':'algebraic','ground_truth':'2.426e-12 * (1 - cos_theta)','variable_descriptions':{'cos_theta':'cos of scattering angle'},'protocol':'A'}))

        elif domain == 'chemistry':
            Temp = np.random.uniform(273,373,N)
            test_cases.append(('Arrhenius Equation: k = 1e11*exp(-80000/(8.314*Temp))', Temp.reshape(-1,1), 1e11*np.exp(-80000/(8.314*Temp)), ['Temp'],
                {'equation_name':'arrhenius_equation','difficulty':'hard','formula_type':'exponential','ground_truth':'1e11 * np.exp(-80000 / (8.314 * Temp))','variable_descriptions':{'Temp':'temperature'},'protocol':'B'}))
            A_m = np.random.uniform(0.01,1,N); HA = np.random.uniform(0.01,1,N)
            test_cases.append(('Henderson-Hasselbalch: pH = 6.5 + log10([A-]/[HA])', np.column_stack([A_m,HA]), 6.5+np.log10(A_m/(HA+1e-12)), ['A_minus','HA'],
                {'equation_name':'henderson_hasselbalch','difficulty':'medium','formula_type':'logarithmic','ground_truth':'6.5 + np.log10(A_minus / HA)','variable_descriptions':{'A_minus':'conjugate base','HA':'weak acid'},'protocol':'B'}))
            E0 = np.random.uniform(0.1,1.5,N); Temp = np.random.uniform(273,373,N); n = np.random.randint(1,3,N).astype(float); Qr = np.random.uniform(0.01,100,N)
            test_cases.append(('Nernst Equation: E = E0 - (8.314*Temp/(n*96485))*ln(Qr)', np.column_stack([E0,Temp,n,Qr]), E0-(8.314*Temp/(n*96485))*np.log(Qr), ['E0','Temp','n','Qr'],
                {'equation_name':'nernst_equation','difficulty':'hard','formula_type':'logarithmic','ground_truth':'E0 - (8.314 * Temp / (n * 96485)) * np.log(Qr)','variable_descriptions':{'E0':'std potential','Temp':'temperature','n':'electrons','Qr':'reaction quotient'},'protocol':'B'}))

        elif domain == 'biology':
            Sub = np.random.uniform(0.1,50,N)
            test_cases.append(('Michaelis-Menten: v = (50*[Sub])/(10+[Sub])', Sub.reshape(-1,1), (50*Sub)/(10+Sub), ['Sub'],
                {'equation_name':'michaelis_menten','difficulty':'medium','formula_type':'rational','ground_truth':'(50.0 * Sub) / (10.0 + Sub)','variable_descriptions':{'Sub':'substrate concentration'},'protocol':'B'}))
            r = np.random.uniform(0.1,0.5,N); Pop = np.random.uniform(10,900,N); K = np.random.uniform(1000,2000,N)
            test_cases.append(('Logistic Growth: dPop/dt = r*Pop*(1-Pop/K)', np.column_stack([r,Pop,K]), r*Pop*(1-Pop/K), ['r','Pop','K'],
                {'equation_name':'logistic_growth','difficulty':'medium','formula_type':'nonlinear','ground_truth':'r * Pop * (1 - Pop / K)','variable_descriptions':{'r':'growth rate','Pop':'population','K':'carrying capacity'},'protocol':'B'}))
            M = np.random.uniform(0.1,100,N)
            test_cases.append(('Allometric Scaling: Y = 3.5*M^0.75', M.reshape(-1,1), 3.5*M**0.75, ['M'],
                {'equation_name':'allometric_scaling','difficulty':'easy','formula_type':'power_law','ground_truth':'3.5 * M**0.75','variable_descriptions':{'M':'body mass'},'protocol':'B'}))

        elif domain == 'mathematics':
            a = np.random.uniform(1,10,N); b = np.random.uniform(1,10,N)
            test_cases.append(('Pythagorean Theorem: c = sqrt(a² + b²)', np.column_stack([a,b]), np.sqrt(a**2+b**2), ['a','b'],
                {'equation_name':'pythagorean_theorem','difficulty':'easy','formula_type':'power_law','ground_truth':'np.sqrt(a**2 + b**2)','variable_descriptions':{'a':'side a','b':'side b'},'protocol':'B'}))
            P = np.random.uniform(1000,10000,N); r = np.random.uniform(0.01,0.1,N); n = np.random.choice([1,4,12],N).astype(float); t = np.random.uniform(1,20,N)
            test_cases.append(('Compound Interest: A = P*(1+r/n)^(n*t)', np.column_stack([P,r,n,t]), P*(1+r/n)**(n*t), ['P','r','n','t'],
                {'equation_name':'compound_interest','difficulty':'medium','formula_type':'exponential','ground_truth':'P * (1 + r/n)**(n*t)','variable_descriptions':{'P':'principal','r':'rate','n':'compounding','t':'years'},'protocol':'B'}))
            a = np.random.uniform(-5,5,N); a[np.abs(a)<0.1]=1.0; b2 = np.random.uniform(-10,10,N); c = np.random.uniform(-5,5,N)
            test_cases.append(('Quadratic Discriminant: Δ = b² - 4ac', np.column_stack([a,b2,c]), b2**2-4*a*c, ['a','b','c'],
                {'equation_name':'quadratic_discriminant','difficulty':'easy','formula_type':'polynomial','ground_truth':'b**2 - 4*a*c','variable_descriptions':{'a':'quadratic coeff','b':'linear coeff','c':'constant'},'protocol':'B'}))

        elif domain == 'economics':
            Q = np.random.uniform(100,1000,N); dQ = np.random.uniform(-50,50,N); P = np.random.uniform(10,100,N); dP = np.random.uniform(-5,5,N)
            dP[np.abs(dP)<0.1] = 0.1
            test_cases.append(('Price Elasticity: Ed = (ΔQ/Q)/(ΔP/P)', np.column_stack([Q,dQ,P,dP]), (dQ/(Q+1e-10))/((dP/(P+1e-10))+1e-10), ['Q','delta_Q','P','delta_P'],
                {'equation_name':'elasticity_demand','difficulty':'medium','formula_type':'rational','ground_truth':'(delta_Q / Q) / (delta_P / P)','variable_descriptions':{'Q':'quantity','delta_Q':'dQ','P':'price','delta_P':'dP'},'protocol':'B'}))
            A = np.random.uniform(1,5,N); K = np.random.uniform(100,1000,N); L = np.random.uniform(10,100,N)
            test_cases.append(('Cobb-Douglas: Y = A*K^0.3*L^0.7', np.column_stack([A,K,L]), A*K**0.3*L**0.7, ['A','K','L'],
                {'equation_name':'cobb_douglas','difficulty':'medium','formula_type':'power_law','ground_truth':'A * K**0.3 * L**0.7','variable_descriptions':{'A':'productivity','K':'capital','L':'labor'},'protocol':'B'}))
            FC = np.random.uniform(10000,100000,N); P2 = np.random.uniform(50,200,N); VC = np.random.uniform(20,100,N)
            test_cases.append(('Break-Even Point: BEP = FC/(P-VC)', np.column_stack([FC,P2,VC]), FC/(P2-VC+1e-10), ['FC','P','VC'],
                {'equation_name':'break_even_point','difficulty':'easy','formula_type':'algebraic','ground_truth':'FC / (P - VC)','variable_descriptions':{'FC':'fixed costs','P':'price','VC':'variable cost'},'protocol':'B'}))

        return test_cases

print(f'ExperimentProtocolAll loaded — {len(ExperimentProtocolAll.get_all_domains())} domains')
total = sum(len(ExperimentProtocolAll.load_test_data(d, num_samples=10))
            for d in ExperimentProtocolAll.get_all_domains())
print(f'Total test cases: {total}')


In [ ]:
# ── SymbolicEngineMethod — LLM-augmented PySR wrapper ────────────────────────
# This is the key Method 5 from the benchmark suite.
# It runs PySR and then asks Claude to refine / interpret the best expression.

import inspect
import anthropic

class MethodResult:
    def __init__(self, success, r2=None, expression=None, error=None, metadata=None):
        self.success    = success
        self.r2         = r2
        self.expression = expression
        self.error      = error
        self.metadata   = metadata or {}

    def to_dict(self):
        return {'success': self.success, 'r2': self.r2,
                'expression': self.expression, 'error': self.error,
                'metadata': self.metadata}


class SymbolicEngineMethod:
    """
    Method 5: PySR symbolic regression + LLM interpretation.
    Mirrors run_exp2_symbolic_engine.py's worker behaviour.
    """
    def __init__(self, verbose=True, pysr_timeout=1100, llm_model='claude-sonnet-4-20250514'):
        self.verbose      = verbose
        self.pysr_timeout = pysr_timeout
        self.llm_model    = llm_model

    # ── PySR helper ──────────────────────────────────────────────────────────
    @staticmethod
    def _make_pysr(seed=42, niterations=1000, timeout_secs=1100, populations=30):
        if not PYSR_AVAILABLE:
            raise RuntimeError('PySR not installed')
        valid = set(inspect.signature(PySRRegressor.__init__).parameters.keys())
        kwargs = dict(
            niterations=niterations, populations=populations, population_size=33,
            maxsize=30, parsimony=0.01,
            binary_operators=['+','-','*','/'],
            unary_operators=['exp','log','sin','cos','sqrt'],
            random_state=seed, verbosity=0, progress=False,
        )
        if 'timeout_in_seconds' in valid: kwargs['timeout_in_seconds'] = timeout_secs
        if 'parallelism'        in valid: kwargs['parallelism'] = 'multithreading'  # multiprocessing → multithreading: avoids Distributed.ProcessExitedException on Kaggle/Julia 1.11
        if 'tournament_selection_n' in valid: kwargs['tournament_selection_n'] = 3
        if 'crossover_probability'  in valid: kwargs['crossover_probability'] = 0.9
        return PySRRegressor(**kwargs)

    # ── safe R² ──────────────────────────────────────────────────────────────
    @staticmethod
    def _safe_r2(y_true, y_pred):
        if y_true is None or y_pred is None or len(y_true) == 0:
            return None
        ss_res = np.sum((y_true - y_pred)**2)
        ss_tot = np.sum((y_true - np.mean(y_true))**2)
        thresh = max(1e-10*(np.max(np.abs(y_true))**2)*len(y_true), 1e-300)
        if ss_tot < thresh:
            return 1.0 if ss_res < 1e-20 else 0.0
        return 1 - ss_res / ss_tot

    # ── LLM refinement ───────────────────────────────────────────────────────
    def _llm_refine(self, description, var_names, best_expr, train_r2):
        """Ask Claude to interpret / simplify the best PySR expression."""
        api_key = os.environ.get('ANTHROPIC_API_KEY', '')
        if not api_key:
            return best_expr, 'no_api_key'
        try:
            client = anthropic.Anthropic(api_key=api_key)
            prompt = (
                f"Equation task: {description}\n"
                f"Variables: {var_names}\n"
                f"PySR found: {best_expr}  (train R²={train_r2:.4f})\n\n"
                "Simplify this expression if possible, or confirm it is already minimal. "
                "Reply with ONLY the simplified expression (Python syntax, no explanation)."
            )
            msg = client.messages.create(
                model=self.llm_model, max_tokens=256,
                messages=[{'role':'user','content':prompt}]
            )
            refined = msg.content[0].text.strip()
            return refined, None
        except Exception as e:
            return best_expr, str(e)

    # ── main entry point ─────────────────────────────────────────────────────
    def run(self, description, X, y, var_names, meta, verbose=True):
        t0 = time.time()
        # Split
        X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

        # PySR
        if not PYSR_AVAILABLE:
            return MethodResult(success=False, error='pysr_not_installed')
        try:
            model = self._make_pysr(timeout_secs=self.pysr_timeout)
            model.fit(X_tr, y_tr)
            best_expr = str(model.sympy())
            r2_train  = self._safe_r2(y_tr, model.predict(X_tr))
            r2_val    = self._safe_r2(y_val, model.predict(X_val))
        except Exception as e:
            return MethodResult(success=False, error=str(e), metadata={'elapsed_s': time.time()-t0})

        # LLM refinement
        refined_expr, llm_err = self._llm_refine(description, var_names, best_expr, r2_train or 0)

        success = (r2_val is not None and r2_val > 0.90)
        return MethodResult(
            success=success, r2=r2_val, expression=refined_expr,
            metadata={
                'pysr_expr': best_expr, 'llm_expr': refined_expr,
                'llm_error': llm_err, 'r2_train': r2_train,
                'elapsed_s': round(time.time()-t0, 2),
            }
        )

print('SymbolicEngineMethod defined OK')


In [ ]:
# ── Worker script (written to tmp .py and executed per equation) ─────────────
# Mirrors the _WORKER_SCRIPT in run_exp2_symbolic_engine.py exactly.

_WORKER_SCRIPT = textwrap.dedent(r"""
import gc, json, os, resource, signal, sys, time, inspect, warnings
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

if len(sys.argv) < 3:
    sys.exit('worker: expected <input_json> <output_json>')

input_path  = Path(sys.argv[1])
output_path = Path(sys.argv[2])
payload     = json.loads(input_path.read_text())

description  = payload['description']
var_names    = payload['var_names']
meta         = payload['meta']
domain       = payload['domain']
pysr_timeout = payload.get('pysr_timeout', 1100)
proc_timeout = payload.get('proc_timeout', pysr_timeout + 120)
ram_limit_gb = payload.get('ram_limit_gb', 0)
llm_model    = payload.get('llm_model', 'claude-sonnet-4-20250514')

# RAM cap
if ram_limit_gb > 0:
    lb = int(ram_limit_gb * 1024**3)
    try: resource.setrlimit(resource.RLIMIT_AS, (lb, lb))
    except Exception as _e: print(f'[worker] RLIMIT_AS not set: {_e}', flush=True)

# Self-destruct timer
def _self_kill(signum, frame):
    print(f'[worker] hard timeout {proc_timeout}s — killing', flush=True)
    os.killpg(os.getpgid(0), signal.SIGKILL)
signal.signal(signal.SIGALRM, _self_kill)
signal.alarm(proc_timeout)

os.environ.setdefault('JULIA_GC_THRESHOLD', '0.4')
os.environ.setdefault('JULIA_NUM_THREADS',  '1')
os.environ.setdefault('JULIA_CPU_THREADS',  '1')
os.environ.setdefault('PYTHON_JULIACALL_HANDLE_SIGNALS', 'yes')
try:
    import juliacall as _jc
except Exception: pass

X = np.array(payload['X'])
y = np.array(payload['y'])

# Load pysr
try:
    from pysr import PySRRegressor
    PYSR_AVAILABLE = True
except ImportError:
    PYSR_AVAILABLE = False

def _safe_r2(yt, yp):
    if yt is None or yp is None or len(yt)==0: return None
    ss_res = np.sum((yt-yp)**2); ss_tot = np.sum((yt-np.mean(yt))**2)
    thresh = max(1e-10*(np.max(np.abs(yt))**2)*len(yt), 1e-300)
    if ss_tot < thresh: return 1.0 if ss_res < 1e-20 else 0.0
    return 1 - ss_res/ss_tot

def _make_pysr(seed=42, niterations=1000, timeout_secs=1100, populations=30):
    valid = set(inspect.signature(PySRRegressor.__init__).parameters.keys())
    kwargs = dict(niterations=niterations, populations=populations, population_size=33,
        maxsize=30, parsimony=0.01, binary_operators=['+','-','*','/'],
        unary_operators=['exp','log','sin','cos','sqrt'],
        random_state=seed, verbosity=0, progress=False)
    if 'timeout_in_seconds' in valid: kwargs['timeout_in_seconds'] = timeout_secs
    if 'parallelism'        in valid: kwargs['parallelism'] = 'multithreading'  # multiprocessing → multithreading: avoids Distributed.ProcessExitedException on Kaggle/Julia 1.11
    if 'tournament_selection_n' in valid: kwargs['tournament_selection_n'] = 3
    if 'crossover_probability'  in valid: kwargs['crossover_probability'] = 0.9
    return PySRRegressor(**kwargs)

t0 = time.time()
success = False; r2_val = None; best_expr = 'N/A'; llm_expr = 'N/A'; error = None

try:
    X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
    model = _make_pysr(timeout_secs=pysr_timeout)
    model.fit(X_tr, y_tr)
    best_expr = str(model.sympy())
    r2_train  = _safe_r2(y_tr, model.predict(X_tr))
    r2_val    = _safe_r2(y_val, model.predict(X_val))

    # LLM refinement
    api_key = os.environ.get('ANTHROPIC_API_KEY','')
    llm_expr = best_expr
    if api_key:
        try:
            import anthropic
            c = anthropic.Anthropic(api_key=api_key)
            prompt = (f'Equation: {description}
Variables: {var_names}
'
                      f'PySR found: {best_expr}  (R²={r2_train:.4f})

'
                      'Simplify if possible. Reply with ONLY the expression (Python syntax).')
            msg = c.messages.create(model=llm_model, max_tokens=256,
                messages=[{'role':'user','content':prompt}])
            llm_expr = msg.content[0].text.strip()
        except Exception as _le:
            print(f'[worker] LLM error: {_le}', flush=True)

    success = (r2_val is not None and r2_val > 0.90)
except Exception as e:
    error = str(e)
    print(f'[worker] ERROR: {e}', flush=True)

elapsed = time.time() - t0
signal.alarm(0)
del X, y; gc.collect()

output_path.write_text(json.dumps({
    'elapsed_s': round(elapsed, 2),
    'result': {
        'success': success, 'r2': r2_val,
        'expression': llm_expr, 'pysr_expr': best_expr,
        'error': error,
    }
}, default=str))
""")

print('Worker script template defined')


In [ ]:
# ── Checkpoint helpers ────────────────────────────────────────────────────────

def _load_checkpoint(path):
    p = Path(path)
    if p.exists():
        try:
            with open(p) as f: return json.load(f)
        except Exception as e:
            print(f'⚠️  Checkpoint read failed: {e}')
    return {'version':1,'method':'SymbolicEngineWithLLM (tools)','completed':[],'results':{}}

def _save_checkpoint(state, path):
    p = Path(path); tmp = p.with_suffix('.tmp')
    with open(tmp,'w') as f: json.dump(state, f, indent=2, default=str)
    os.replace(tmp, p)

def _eq_key(meta, domain):
    return f"{domain}::{meta.get('equation_name', meta.get('name', str(meta)))}"

print('Checkpoint helpers defined')


In [ ]:
# ── Per-equation subprocess runner ───────────────────────────────────────────

def _run_one_equation(description, X, y, var_names, meta, domain,
                      pysr_timeout=PYSR_TIMEOUT, ram_limit_gb=0):
    """Spawn an isolated subprocess; fully mirrors run_exp2_symbolic_engine.py."""
    proc_timeout   = pysr_timeout + 120
    parent_timeout = proc_timeout + 30

    with tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False) as f: inp  = f.name
    with tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False) as f: out  = f.name
    with tempfile.NamedTemporaryFile(mode='w', suffix='.py',   delete=False) as f:
        wk = f.name; f.write(_WORKER_SCRIPT)

    payload = {
        'description': description, 'var_names': var_names,
        'meta': meta, 'domain': domain,
        'pysr_timeout': pysr_timeout, 'proc_timeout': proc_timeout,
        'ram_limit_gb': ram_limit_gb,
        'llm_model': os.environ.get('LLM_MODEL', 'claude-sonnet-4-20250514'),
        'X': X.tolist(), 'y': y.tolist(),
    }
    with open(inp,'w') as f: json.dump(payload, f)

    t0 = time.time(); proc = None
    try:
        proc = subprocess.Popen(
            [sys.executable, wk, inp, out],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1, start_new_session=True,
        )
        for line in proc.stdout: print('  │ '+line, end='', flush=True)
        proc.wait(timeout=parent_timeout)
        elapsed = time.time()-t0
        if proc.returncode != 0:
            print(f'\n  ⚠️  Worker exited {proc.returncode}'); return None, elapsed
        out_text = Path(out).read_text().strip()
        if not out_text:
            print('\n  ⚠️  Worker produced no output'); return None, elapsed
        wr = json.loads(out_text)
        return wr['result'], wr['elapsed_s']
    except subprocess.TimeoutExpired:
        elapsed = time.time()-t0
        if proc:
            try: os.killpg(os.getpgid(proc.pid), signal.SIGKILL)
            except OSError: proc.kill()
            proc.wait()
        print(f'\n  ⏱  Killed after {elapsed:.0f}s'); return None, elapsed
    except Exception as exc:
        elapsed = time.time()-t0
        if proc:
            try: os.killpg(os.getpgid(proc.pid), signal.SIGKILL)
            except OSError: proc.kill()
            proc.wait()
        print(f'\n  ⚠️  Exception: {exc}'); return None, elapsed
    finally:
        for p in (inp, out, wk):
            try: os.unlink(p)
            except OSError: pass

print('Subprocess runner defined')


In [ ]:
# ── Partial-results scoreboard ────────────────────────────────────────────────

def _print_scoreboard(rows, total, suite_start):
    done = len(rows)
    solved  = sum(1 for r in rows if r['status'] == '✅')
    failed  = sum(1 for r in rows if r['status'] in ('❌','⏱','⚠️'))
    skipped = sum(1 for r in rows if r['status'] == '↩')
    wall    = time.time()-suite_start
    if done > 0 and (total-done) > 0:
        eta_s = int(wall/done*(total-done)); h,rem = divmod(eta_s,3600); m,s = divmod(rem,60)
        eta = f'ETA ≈ {h:02d}:{m:02d}:{s:02d}'
    else: eta = 'ETA ≈ —'
    on_track = '✅ on track' if (done==0 or solved/done >= CFG['pass_threshold']/total) else '⚠️  behind'
    BAR = '─'*72; DBAR = '━'*72
    print(f'\n{DBAR}'); print(f'  PARTIAL [{done}/{total}]  {solved} solved  {failed} failed  {skipped} resumed  {eta}'); print(DBAR)
    print(f"  {'#':<4} {'Equation':<26} {'Domain':<12} {'R²':>8}  {'Time':>6}  Status")
    print(f'  {BAR}')
    for i, row in enumerate(rows, 1):
        r2s = f"{row['r2']:.4f}" if row.get('r2') is not None else '  —   '
        ts  = f"{int(row.get('elapsed_s',0))}s" if row.get('elapsed_s') else '  —'
        print(f"  {i:<4} {row['eq_name'][:26]:<26} {row['domain'][:12]:<12} {r2s:>8}  {ts:>6}  {row['status']}")
    print(f'  {BAR}')
    pct = f'{solved/total*100:.1f}%' if total else '—'
    print(f'  Threshold: {CFG["pass_threshold"]}/{total}  |  {solved}/{total} ({pct})  |  {on_track}')
    print(f'{DBAR}\n')

print('Scoreboard defined')


In [ ]:
# ── MAIN LOOP — Run all 30 equations ─────────────────────────────────────────
# Checkpointed after every equation.  Re-run cell to resume.

protocol = ExperimentProtocolAll()
state    = _load_checkpoint(CFG['checkpoint']) if CFG['resume'] else {
    'version':1,'method':'SymbolicEngineWithLLM (tools)','completed':[],'results':{}}

completed = set(state.get('completed',[]))
results   = state.get('results',{})

# Collect all 30 test stubs
all_tests = []
for domain in protocol.get_all_domains():
    for (desc, X, y, var_names, meta) in protocol.load_test_data(domain, num_samples=CFG['num_samples']):
        all_tests.append((desc, var_names, meta, domain))
        del X, y
gc.collect()

if CFG['one_equation']:
    all_tests = [t for t in all_tests if CFG['one_equation'].lower() in t[0].lower()]
    print(f'Filtered to 1 equation: {all_tests[0][0] if all_tests else "NOT FOUND"}')

total = len(all_tests)
print(f'\n🔬  SymbolicEngineWithLLM — {total} equations')
if CFG['resume'] and completed:
    print(f'♻️  Resuming — {len(completed)} done, {total-len(completed)} remaining')
print(f'📋  Checkpoint → {CFG["checkpoint"]}\n')
print('🧠  Memory strategy: each equation runs in an isolated subprocess.\n')

suite_start = time.time(); solved = 0
completed_rows = []

# Pre-populate scoreboard from checkpoint
for key in list(completed):
    r = results.get(key,{}); res = r.get('result') or {}
    r2 = res.get('r2') if res.get('success') else None
    if res.get('success'): solved += 1
    completed_rows.append({'eq_name':r.get('description',key.split('::',1)[-1])[:26],
        'domain':r.get('domain','—'),'r2':r2,'elapsed_s':r.get('elapsed_s'),'status':'↩'})

for i, (description, var_names, meta, domain) in enumerate(all_tests, 1):
    key = _eq_key(meta, domain)
    if key in completed:
        print(f'  ⏭️  SKIP {i}/{total}: {meta.get("equation_name", description)}', flush=True)
        continue
    print(f'\n{"="*72}'); print(f'  [{i}/{total}]  {description}')
    print(f'  Domain: {domain}  |  vars: {var_names}'); print(f'{"="*72}')

    loaded = protocol.load_test_data(domain, num_samples=CFG['num_samples'])
    match  = next((c for c in loaded if c[4].get('equation_name',c[0])==meta.get('equation_name',description) or c[0]==description), None)
    if match is None:
        print('  ⚠️  Could not reload X/y — skipping')
        results[key]={'description':description,'domain':domain,'result':None}
        completed.add(key); state['completed']=list(completed); state['results']=results
        _save_checkpoint(state, CFG['checkpoint'])
        completed_rows.append({'eq_name':meta.get('equation_name',description)[:26],'domain':domain,'r2':None,'elapsed_s':None,'status':'⚠️'})
        _print_scoreboard(completed_rows, total, suite_start); continue

    _, X, y, _, _ = match; del match, loaded

    r_dict, elapsed = _run_one_equation(
        description, X, y, var_names, meta, domain, CFG['pysr_timeout'], CFG['ram_limit_gb'])
    del X, y; gc.collect()

    success  = bool(r_dict and r_dict.get('success'))
    r2_val   = r_dict.get('r2') if success else None
    err_msg  = (r_dict or {}).get('error','worker crash')

    results[key]={'description':description,'domain':domain,'elapsed_s':round(elapsed,2),'result':r_dict}
    completed.add(key)
    state['completed']=list(completed); state['results']=results
    state['timestamp']=datetime.now().isoformat()
    _save_checkpoint(state, CFG['checkpoint'])

    if success: solved += 1; status_str=f'R²={r2_val:.4f}'
    else:        status_str=f'✗ {err_msg or "failed"}'
    print(f'\n  ✔  {meta.get("equation_name",description)} → {status_str}  ({elapsed:.0f}s)', flush=True)

    timed_out  = elapsed >= METHOD_TIMEOUT-5
    row_status = '✅' if success else ('⏱' if timed_out else '❌')
    completed_rows.append({'eq_name':meta.get('equation_name',description)[:26],
        'domain':domain,'r2':r2_val,'elapsed_s':round(elapsed,1),'status':row_status})
    _print_scoreboard(completed_rows, total, suite_start)

wall = time.time()-suite_start; h,rem = divmod(int(wall),3600); m,s = divmod(rem,60)
print(f'\n{"="*72}')
print(f'  DONE — {solved}/{total} solved  |  wall time {h:02d}:{m:02d}:{s:02d}')
print(f'  Checkpoint → {CFG["checkpoint"]}')


In [ ]:
# ── Statistical analysis ──────────────────────────────────────────────────────
import pandas as pd

r2_all     = [r['result']['r2'] for r in results.values() if r.get('result') and r['result'].get('r2') is not None]
successes  = {k:r for k,r in results.items() if r.get('result') and r['result'].get('success')}
r2_succ    = [r['result']['r2'] for r in successes.values()]
n_total    = len(results); n_succ = len(successes)

print('='*65); print('RESULTS SUMMARY'); print('='*65)
if r2_all:
    print(f'SymbolicEngine  n={len(r2_all)}  mean={np.mean(r2_all):.4f}  median={np.median(r2_all):.4f}  success(>0.90)={n_succ}/{n_total} ({n_succ/n_total*100:.1f}%)')

rows = []
for k,r in results.items():
    res = r.get('result') or {}
    domain, desc = r.get('domain','—'), r.get('description','—')
    r2  = res.get('r2'); expr = res.get('expression','—')
    win = r2 is not None and r2 > 0.90
    rows.append({'key':k,'domain':domain,'description':desc[:50],
                 'R²':f'{r2:.4f}' if r2 is not None else '—',
                 'R²_num':r2,'expression':expr[:60],'result':'✓ Pass' if win else '✗ Fail'})

df = pd.DataFrame(rows).sort_values('R²_num', ascending=False)
try:
    display(df[['domain','description','R²','expression','result']].style.applymap(
        lambda v: 'background:#d4edda;color:#155724;font-weight:bold' if '✓' in str(v) else
                  ('background:#f8d7da;color:#721c24' if '✗' in str(v) else ''),
        subset=['result']))
except Exception:
    print(df[['domain','description','R²','expression','result']].to_string(index=False))


In [ ]:
# ── Export results ────────────────────────────────────────────────────────────
import zipfile, os
from datetime import datetime

ts = datetime.now().strftime('%Y%m%d_%H%M%S')
export_dir = f'/hypatiax/data/results/sym_engine_exp2_{ts}'
os.makedirs(export_dir, exist_ok=True)

# JSON
with open(f'{export_dir}/results.json','w') as f: json.dump(results, f, indent=2, default=str)
# CSV
df.to_csv(f'{export_dir}/results.csv', index=False)
# Checkpoint copy
import shutil; shutil.copy(CFG['checkpoint'], f'{export_dir}/checkpoint.json')

zip_path = f'/hypatiax/data/results/sym_engine_exp2_{ts}.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as zf:
    for fn in os.listdir(export_dir): zf.write(os.path.join(export_dir,fn), fn)
print(f'Zip: {zip_path}  ({os.path.getsize(zip_path):,} bytes)')
print('Download via Output tab → Download all')
